# Drive repo fix + Probe 5b + Probe 3b training curve

Run cells **in order**. Part A syncs Drive with GitHub; Part B migrates old checkpoint names; Parts C–D run new probes; Part E downloads results (no Colab git).


## PART A: Fix the Drive repo

### Cell A1 — mount

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
REPO = "/content/drive/MyDrive/vlm-ocr-eval"
%cd $REPO
!pwd

### Cell A2 — kill the phantom file-mode change

Should now print nothing (or only `??` untracked lines). This is the fix for the rebase failures.

In [ ]:
%cd $REPO
!git config core.filemode false
!git status --short

### Cell A3 — see where you are

In [ ]:
%cd $REPO
!git fetch origin
!git log --oneline -1
!git log --oneline -1 origin/main
!git status --short

### Cell A4 — reset Drive to match GitHub

This throws away the Drive-local commit `8271bfa` "Add Hindi VLM OCR probe results" — that's fine, you already have those 19 files on your laptop from the zip. It also clears accumulated stashes. Your `.pt` checkpoints are untracked, so `reset --hard` does not touch them.

In [ ]:
%cd $REPO
!git stash clear
!git reset --hard origin/main
!git log --oneline -3

### Cell A5 — confirm you now have the new code

You **must** see `--script` and `--keep-snapshots` in the output. If you don't, stop — everything below depends on it.

In [ ]:
%cd $REPO
!python src/models/instrument/train.py --help

### Cell A6 — install dependencies

In [ ]:
%cd $REPO
!pip install -q -r requirements.txt uharfbuzz aksharamukha
!apt-get update -qq && apt-get install -y -qq fonts-noto-core fonts-noto-extra
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## PART B: Make your existing checkpoints work with the new code

Your 9 checkpoints are named the old way and are missing the `script` field the new safety guards check for. This fixes both **without retraining**.

### Cell B1 — rename + inject the script field

In [ ]:
import torch, shutil
from pathlib import Path

CKPT = Path("/content/drive/MyDrive/vlm-ocr-eval/checkpoints")
BACKUP = Path("/content/drive/MyDrive/checkpoints_backup_prerename")
if not BACKUP.exists():
    shutil.copytree(CKPT, BACKUP)
    print(f"Backup made at {BACKUP}")
else:
    print("Backup already exists, skipping")

for cond in ["natural", "flattened", "inverted"]:
    for seed in [0, 1, 2]:
        old = CKPT / f"checkpoint_{cond}_seed{seed}.pt"
        new = CKPT / f"checkpoint_hindi_{cond}_seed{seed}.pt"
        if new.exists():
            print(f"skip (exists): {new.name}")
            continue
        if not old.exists():
            print(f"MISSING: {old.name}")
            continue
        ck = torch.load(old, map_location="cpu")
        ck["script"] = "hindi"          # new guards require this
        torch.save(ck, new)
        print(f"{old.name} -> {new.name}  (step {ck.get('step')})")

    old_tok = CKPT / f"tokenizer_{cond}.json"
    new_tok = CKPT / f"tokenizer_hindi_{cond}.json"
    if old_tok.exists() and not new_tok.exists():
        shutil.copy2(old_tok, new_tok)
        print(f"{old_tok.name} -> {new_tok.name}")

### Cell B2 — verify

Expect 9 `checkpoint_hindi_*.pt` and 3 `tokenizer_hindi_*.json`.

In [ ]:
from pathlib import Path
CKPT = Path("/content/drive/MyDrive/vlm-ocr-eval/checkpoints")
for p in sorted(CKPT.glob("*hindi*")):
    print(f"{p.name:45s} {p.stat().st_size/1024**2:7.1f} MB")

## PART C: Probe 5b (the zero-shot floor)

Hindi-trained model on Ol Chiki (Santhali) and Perso-Arabic (Kashmiri) — scripts it has **never** seen.

### Cell C1 — preflight

All must be True / non-zero. If `probe5b` code is False, push from laptop first.

In [ ]:
from pathlib import Path
REPO = Path("/content/drive/MyDrive/vlm-ocr-eval")
print("probe5b code:", (REPO/"src/probes/probe5b_zeroshot_floor.py").exists())
for lang in ("hindi", "santhali", "kashmiri"):
    gt = REPO/"data/raw"/lang/"ground_truth.jsonl"
    n = len(list((REPO/"data/raw"/lang/"images").glob("*.png"))) if (REPO/"data/raw"/lang/"images").exists() else 0
    print(f"{lang}: gt={gt.exists()}  images={n}")

### Cell C2 — check flags, then run

In [ ]:
%cd $REPO
!python src/probes/probe5b_zeroshot_floor.py --help

In [ ]:
%cd $REPO
!python src/probes/probe5b_zeroshot_floor.py \
    --script hindi --condition natural --seed 0 \
    --output-root /content/drive/MyDrive/vlm-ocr-eval/checkpoints \
    --n-samples 100 \
    --out data/probe_results/probe5b_hindi_natural_seed0.jsonl \
    --device cuda

## PART D: Probe 3b (training curve)

Needs a **fresh** training run with snapshots (~17 min). Old checkpoints overwrote intermediates.

### Cell D1 — retrain natural/seed0 with snapshots

If it says "resuming at step 5000" and finishes instantly, Cell B1 already created that checkpoint. To force a real fresh run, run the delete cell below first, then re-run training. Your backup from B1 still has the original.

In [ ]:
%cd $REPO
!python src/models/instrument/train.py \
    --manifest data/manifests/hindi_natural.jsonl \
    --output-root /content/drive/MyDrive/vlm-ocr-eval/checkpoints \
    --script hindi --condition natural --seed 0 \
    --batch-size 32 --total-steps 5000 \
    --checkpoint-every 500 --keep-snapshots

**Optional — force fresh training** (delete main checkpoint first):

In [ ]:
!rm -f $REPO/checkpoints/checkpoint_hindi_natural_seed0.pt

### Cell D2 — check snapshots landed

In [ ]:
from pathlib import Path
CKPT = Path("/content/drive/MyDrive/vlm-ocr-eval/checkpoints")
for p in sorted(CKPT.glob("*step*.pt")):
    print(p.name)

### Cell D3 — run the curve probe

In [ ]:
%cd $REPO
!python src/probes/probe3_training_curve.py \
    --manifest data/manifests/hindi_natural.jsonl \
    --output-root /content/drive/MyDrive/vlm-ocr-eval/checkpoints \
    --script hindi --condition natural --seed 0 \
    --steps 500,1000,2000,3000,5000 \
    --out data/probe_results/probe3_curve_hindi_natural_seed0.json \
    --device cuda

## PART E: Get results out — no git from Colab

Skip Colab git entirely. Download the zip to your laptop and unpack into `data/probe_results/`.

### Cell E1

In [ ]:
%cd $REPO
!rm -f /content/new_results.zip
!zip -q -j /content/new_results.zip \
    data/probe_results/probe5b_hindi_natural_seed0.jsonl \
    data/probe_results/probe3_curve_hindi_natural_seed0.json
!ls -lh /content/new_results.zip
from google.colab import files
files.download("/content/new_results.zip")